# VigiaEnchente — MVP Machine Learning

Modelo de previsao de risco de enchente para Sabara/MG usando Random Forest.

**Pipeline completo:** preparacao de dados → treinamento → avaliacao

---
## PARTE 1: Preparacao dos Dados

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns


RAW = Path("data_csv/raw")
PROCESSED = Path("data_csv/processed")
MODELS = Path("models")
OUTPUTS = Path("outputs")
for p in [PROCESSED, MODELS, OUTPUTS]:
    p.mkdir(parents=True, exist_ok=True)
print("OK")

### 1.1 Carregar Chuvas (ANA - Estacao 1943006)

In [ ]:
chuvas_raw = pd.read_csv(RAW / "chuvas.csv", sep=";", decimal=",", skiprows=13, encoding="latin-1")
print(f"Registros mensais: {len(chuvas_raw)}")

### 1.2 Despivotar

In [ ]:
def despivotar_chuvas(df):
    registros = []
    for _, row in df.iterrows():
        data_ref = pd.to_datetime(row["Data"], dayfirst=True)
        ano = data_ref.year
        mes = data_ref.month
        nivel = row["NivelConsistencia"]
        for dia in range(1, 32):
            col = f"Chuva{dia:02d}"
            if col not in row.index:
                continue
            valor = row[col]
            try:
                data = pd.Timestamp(year=ano, month=mes, day=dia)
            except ValueError:
                continue
            registros.append({"data": data, "chuva_mm": valor if pd.notna(valor) else np.nan, "nivel_consistencia": nivel})
    return pd.DataFrame(registros)

print("Despivotando...")
chuvas_diaria = despivotar_chuvas(chuvas_raw)
print(f"Registros diarios: {len(chuvas_diaria)}")

### 1.3 Filtrar 1997-2025

In [ ]:
chuvas_diaria = chuvas_diaria.sort_values("nivel_consistencia", ascending=False)
chuvas_diaria = chuvas_diaria.drop_duplicates(subset="data", keep="first")
chuvas_diaria = chuvas_diaria.sort_values("data").reset_index(drop=True)
chuvas_diaria = chuvas_diaria[(chuvas_diaria["data"] >= "1997-01-01") & (chuvas_diaria["data"] <= "2025-12-31")].reset_index(drop=True)
print(f"Dias: {len(chuvas_diaria)}")

### 1.4 Carregar Dados Externos

- **GloFAS** (vazao simulada): modelo hidrologico LISFLOOD/ECMWF via `flood-api.open-meteo.com`
- **ERA5** (meteorologia): reanalise atmosferica ECMWF via `archive-api.open-meteo.com`

In [ ]:
glofas = pd.read_csv(RAW / "glofas_vazao.csv", parse_dates=["data"])
glofas = glofas.rename(columns={"vazao_glofas_m3s": "vazao"})

meteo = pd.read_csv(RAW / "meteo_historico.csv", parse_dates=["data"])

print(f"GloFAS (vazao): {len(glofas)} dias")
print(f"ERA5 (meteo): {len(meteo)} dias | Variaveis: {list(meteo.columns[1:])}")

### 1.5 Merge + Features

In [ ]:
df = pd.merge(chuvas_diaria[["data", "chuva_mm"]], glofas, on="data", how="inner")
df = pd.merge(df, meteo, on="data", how="inner")
df = df.sort_values("data").reset_index(drop=True)
df["chuva_mm"] = df["chuva_mm"].fillna(0)

df["acumulado_3d"] = df["chuva_mm"].rolling(window=3, min_periods=1).sum()
df["acumulado_7d"] = df["chuva_mm"].rolling(window=7, min_periods=1).sum()
df["chuva_ontem"] = df["chuva_mm"].shift(1)
df["chuva_anteontem"] = df["chuva_mm"].shift(2)
df["chuva_3d_atras"] = df["chuva_mm"].shift(3)
df["vazao_ontem"] = df["vazao"].shift(1)
df["vazao_anteontem"] = df["vazao"].shift(2)

print(f"Registros: {len(df)} | Features criadas.")

### 1.6 Target

Criterio combinado:
- `vazao D+1 >= 7.5 m3/s` (inundacao fluvial)
- `acumulado_3d D+1 >= 100mm` (chuva extrema)

Target = 1 se qualquer condicao for verdadeira no dia seguinte.

In [ ]:
df["acumulado_3d_amanha"] = df["acumulado_3d"].shift(-1)
df["vazao_amanha"] = df["vazao"].shift(-1)

df["target"] = ((df["vazao_amanha"] >= 7.5) | (df["acumulado_3d_amanha"] >= 100)).astype(int)

print(df["target"].value_counts())
print(f"Dias com risco: {df['target'].mean()*100:.2f}%")

### 1.7 Salvar Base Final

In [ ]:
df_final = df.dropna(subset=["target", "chuva_3d_atras", "vazao_anteontem"]).reset_index(drop=True)
df_final["target"] = df_final["target"].astype(int)

colunas = ["data", "chuva_mm", "acumulado_3d", "acumulado_7d",
           "chuva_ontem", "chuva_anteontem", "chuva_3d_atras",
           "vazao", "vazao_ontem", "vazao_anteontem",
           "temp_max", "temp_min", "umidade_media",
           "vento_max", "evapotranspiracao", "pressao_media", "target"]
df_final = df_final[colunas].round(2)

df_final.to_csv(PROCESSED / "base_modelo.csv", index=False)
print(f"Base: {len(df_final)} registros | Features: {len(colunas)-2}")
print(f"Target: {df_final['target'].value_counts().to_dict()}")

### 1.8 Eventos Confirmados

Setar target=1 nas datas do PLANCON e salvar tabela de eventos (extraida da base final).

In [ ]:
# Datas dos eventos confirmados (PLANCON Sabara 2025/2028)
datas_eventos = pd.to_datetime(["1997-12-14", "2020-01-27", "2022-01-09", "2023-10-26", "2024-11-13"])

# Setar target=1 na base final
df_final.loc[df_final["data"].isin(datas_eventos), "target"] = 1
df_final.to_csv(PROCESSED / "base_modelo.csv", index=False)

# Extrair tabela de eventos da base final e salvar
eventos_tabela = df_final[df_final["data"].isin(datas_eventos)]
eventos_tabela.to_csv(PROCESSED / "eventos_confirmados.csv", index=False)
print(eventos_tabela.to_string(index=False))

---
## PARTE 2: Treinamento

### 2.1 Split Temporal

In [ ]:
features = [c for c in df_final.columns if c not in ["data", "target"]]

treino = df_final[df_final["data"] < "2018-01-01"].copy()
validacao = df_final[(df_final["data"] >= "2018-01-01") & (df_final["data"] < "2020-01-01")].copy()
teste = df_final[df_final["data"] >= "2020-01-01"].copy()

X_treino = treino[features]
y_treino = treino["target"]
X_val = validacao[features]
y_val = validacao["target"]
X_teste = teste[features]
y_teste = teste["target"]

print(f"Treino: {len(treino)} dias | Target=1: {y_treino.sum()} ({y_treino.mean()*100:.1f}%)")
print(f"Validacao: {len(validacao)} dias | Target=1: {y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"Teste: {len(teste)} dias | Target=1: {y_teste.sum()} ({y_teste.mean()*100:.1f}%)")
print(f"Features ({len(features)}): {features}")

### 2.2 Treinar Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf.fit(X_treino, y_treino)
print(f"Score treino: {rf.score(X_treino, y_treino):.4f}")
print(f"Score validacao: {rf.score(X_val, y_val):.4f}")

### 2.3 Baseline Regra de Chuva (100mm/72h)

In [ ]:
teste["pred_regra_chuva"] = (teste["acumulado_3d"] >= 100).astype(int)
teste["pred_rf"] = rf.predict(X_teste)
teste["prob_rf"] = rf.predict_proba(X_teste)[:, 1]

print(f"RF previu risco: {teste['pred_rf'].sum()} dias")
print(f"Regra de Chuva previu risco: {teste['pred_regra_chuva'].sum()} dias")
print(f"Risco real: {y_teste.sum()} dias")

### 2.4 Salvar Modelo

In [ ]:
with open(MODELS / "random_forest.pkl", "wb") as f:
    pickle.dump(rf, f)
print("Modelo salvo.")

---
## PARTE 3: Avaliacao

### 3.1 Metricas

In [ ]:
# Metricas na VALIDACAO (2018-2019)
validacao["pred_rf"] = rf.predict(X_val)
validacao["pred_regra_chuva"] = (validacao["acumulado_3d"] >= 100).astype(int)

print("VALIDACAO (2018-2019):")
print("\nRANDOM FOREST:")
print(classification_report(y_val, validacao["pred_rf"], target_names=["Normal", "Risco"]))
print("REGRA DE CHUVA (100mm/72h):")
print(classification_report(y_val, validacao["pred_regra_chuva"], target_names=["Normal", "Risco"]))

# Metricas no TESTE (2020-2025)
y_real = teste["target"]
y_rf = teste["pred_rf"]
y_regra = teste["pred_regra_chuva"]

print("\nTESTE (2020-2025):")
print("\nRANDOM FOREST:")
print(classification_report(y_real, y_rf, target_names=["Normal", "Risco"]))
print("REGRA DE CHUVA (100mm/72h):")
print(classification_report(y_real, y_regra, target_names=["Normal", "Risco"]))

### 3.2 Matriz de Confusao

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, y_pred, titulo in [(axes[0], y_rf, "Random Forest"), (axes[1], y_regra, "Regra de Chuva")]:
    cm = confusion_matrix(y_real, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["Normal", "Risco"], yticklabels=["Normal", "Risco"])
    ax.set_title(titulo)
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.savefig(OUTPUTS / "matriz_confusao.png", dpi=100, bbox_inches="tight")
plt.show()

### 3.3 Validacao em TODOS os Eventos

In [ ]:
df_final["pred_rf"] = rf.predict(df_final[features])
df_final["prob_rf"] = rf.predict_proba(df_final[features])[:, 1]
df_final["pred_regra"] = (df_final["acumulado_3d"] >= 100).astype(int)

val_all = df_final[df_final["data"].isin(datas_eventos)].copy()
val_all["rf_acertou"] = val_all["pred_rf"].map({1: "SIM", 0: "NAO"})
val_all["regra_acertou"] = val_all["pred_regra"].map({1: "SIM", 0: "NAO"})

print("TODOS OS EVENTOS CONFIRMADOS:")
print(val_all[["data", "prob_rf", "rf_acertou", "regra_acertou"]].to_string(index=False))
print(f"\nRF: {(val_all['pred_rf']==1).sum()}/{len(val_all)}")
print(f"Regra da chuva: {(val_all['pred_regra']==1).sum()}/{len(val_all)}")

### 3.4 Feature Importance

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importancias.plot(kind="barh", color="steelblue")
plt.title("Importancia das Features - Random Forest")
plt.xlabel("Importancia")
plt.tight_layout()
plt.savefig(OUTPUTS / "feature_importance.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nRanking:")
print(importancias.sort_values(ascending=False).to_string())

### 3.5 Resultado Final

In [ ]:
f1_rf = f1_score(y_real, y_rf)
f1_regra = f1_score(y_real, y_regra)

print("=" * 50)
print("RESULTADO FINAL")
print("=" * 50)
print(f"F1 Random Forest:    {f1_rf:.4f}")
print(f"F1 Regra de Chuva:   {f1_regra:.4f}")
if f1_rf > f1_regra:
    diff_pct = ((f1_rf - f1_regra) / f1_regra) * 100
    print(f"\nRF supera Regra de Chuva em {diff_pct:.1f}%")
    print(f"Calculo: ({f1_rf:.4f} - {f1_regra:.4f}) / {f1_regra:.4f} * 100 = {diff_pct:.1f}%")
else:
    diff_pct = ((f1_regra - f1_rf) / f1_rf) * 100
    print(f"\nRegra supera RF em {diff_pct:.1f}%")
    print(f"Calculo: ({f1_regra:.4f} - {f1_rf:.4f}) / {f1_rf:.4f} * 100 = {diff_pct:.1f}%")